In [2]:
%load_ext autoreload
%autoreload 2

****************************************
# Galactic: Crab Analysis (DC1 and DC2)

This notebook analyzes the Crab source using both the **DC1-style Crab simulation** and the newer **DC2 Crab simulation**.

The purpose is to compare the two analysis cases within the DC2 framework while reusing a familiar source.

For **Crab DC1**, the goals are to:

- Localize the source
- Measure the spectrum
- Reconstruct an image of the source

For **Crab DC2**, the goal is to:

- Detect the source
- Measure the phase-integrated spectrum

The DC2 Crab simulation includes the combined **pulsar + nebula spectrum**, divided into five energy bands from **100 keV to 10 MeV**. 
**************

## set up

### imports

In [3]:
import numpy as np
import pandas as pd 
import cosipy as cp 

from pathlib import Path
import sys

repo_root = Path.cwd().parent

import astropy.units as u
import matplotlib.pyplot as plt

from threeML import Band, PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter, Powerlaw

from astropy.io import fits

%matplotlib inline

### data

Define file names

In [6]:
data_dir = Path(repo_root, "data")

data_crab_dc1 = data_dir / "data-products/downloads/crab_3months_unbinned_data.fits.gz"
data_crab_dc2 = data_dir / "data-products/downloads/Crab_DC2_3months_unbinned_data.fits.gz"
data_orientation = data_dir / "data-products/downloads/20280301_3_month.ori"

data_response = data_dir / (
    "data-products/downloads/"
    "SMEXv12.Continuum.HEALPixO3_10bins_log_flat."
    "binnedimaging.imagingresponse.nonsparse_nside8."
    "area.good_chunks_unzip.h5.zip"
)

data_background = data_dir / "backgrounds/downloads/total_bg_3months_unbinned_data.fits.gz"

In [21]:
# Paths to the binned data files
crab_dc1_binned = repo_root / "data" / "data-products" / "binned" / "crab_binned_data_dc1.hdf5"
crab_dc2_binned = repo_root / "data" / "data-products" / "binned" / "crab_binned_data_dc2.hdf5"
background_binned = repo_root / "data" / "backgrounds" / "binned" / "background_binned_data.hdf5"

In [5]:

# Checking the contents of the FITS file for YAML file creation
with fits.open(data_crab_dc1) as hdul:
    hdul.info()
    print(hdul[1].columns.names)
    # Checking the time range of the data
    times = hdul[1].data["TimeTags"]
# Obtaining the minimum and maximum time values from the TimeTags column
print('tmin:', times.min(), 'tmax:', times.max())

Filename: /Users/maura/Documents/COSI/data-challenge-2/data/data-products/downloads/crab_3months_unbinned_data.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     46   6432442R x 11C   [D, D, 2D, 2D, 2D, D, D, D, D, D, D]   
['Energies', 'TimeTags', 'Xpointings (glon,glat)', 'Ypointings (glon,glat)', 'Zpointings (glon,glat)', 'Phi', 'Chi local', 'Psi local', 'Distance', 'Chi galactic', 'Psi galactic']
tmin: 1835487684.5801868 tmax: 1843466661.583874


### Inputs

In [10]:
l_crab,b_crab = 184.55746, -5.78436 # Galactic longitude & latitude of Crab

ul = 3 # SNR limit for upper limits on spectral fit

***
## Binning the Data

### DC1, DC2 & background binning

In [11]:
# In this part, it seems that it read the intups.yaml (this is the file that contains the inputs for the analysis.)
# In data-challenge-1 this was done in the notebook, but in data-challenge-2 it is done in the inputs.yaml file. (automated)
analysis1 = cp.BinnedData("inputs.yaml")
analysis2 = cp.BinnedData("inputs.yaml")
analysisbg = cp.BinnedData("inputs.yaml")



In [24]:
# Create the DC1 binned data only if the file does not already exist
if not crab_dc1_binned.exists():
    analysis1.get_binned_data(
        unbinned_data=str(data_crab_dc1),
        output_name=str(crab_dc1_binned.with_suffix(""))
    )

# Create the DC2 binned data only if the file does not already exist
if not crab_dc2_binned.exists():
    analysis2.get_binned_data(
        unbinned_data=str(data_crab_dc2),
        output_name=str(crab_dc2_binned.with_suffix(""))
    )

if not background_binned.exists():
    analysisbg.get_binned_data(
        unbinned_data=str(data_background),
        output_name=str(background_binned.with_suffix(""))
    )

    analysisbg.load_binned_data_from_hdf5(
    binned_data=str(background_binned)
    )

In [28]:
# are our data files created?
print("Crab DC1 binned data exists:", crab_dc1_binned.exists())
print("Crab DC2 binned data exists:", crab_dc2_binned.exists()) 
print("Background binned data exists:", background_binned.exists())

Crab DC1 binned data exists: True
Crab DC2 binned data exists: True
Background binned data exists: True


### Exploring raw binned data: File shape & plots

In [27]:
# lets see the shape of the binned data
print("Crab DC1 binned data shape:", analysis1.binned_data.shape)
print("Crab DC2 binned data shape:", analysis2.binned_data.shape)
print("Background binned data shape:", analysisbg.binned_data.shape) 

Crab DC1 binned data shape: (1108, 10, 36, 768)
Crab DC2 binned data shape: (1108, 10, 36, 768)
Background binned data shape: (1108, 10, 36, 768)
